# ToneFit ML — Google Colab Pipeline

**FaRL-64 Baseline vs Hierarchical FaRL-64**  
Dataset: Deep Armocromia (Stacchio et al., ECCV 2024)

---

**Before running:**
1. Go to `Runtime → Change runtime type → T4 GPU`
2. Mount Google Drive (Step 1) — dataset must be uploaded to Drive first

**Pipeline:**
- **Step 6** — Train FaRL-64 Baseline (Model A: joint 4-season + 12-subtype)
- **Step 7** — Train Hierarchical FaRL-64 (novel contribution)
- **Step 8** — Compare all models side by side
- **Step 9** — Save results to Drive

---
## Step 0 — Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f'✅ GPU available: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('❌ No GPU detected!')
    print('Go to Runtime → Change runtime type → T4 GPU, then re-run.')

---
## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

---
## Step 2 — Clone Repo & Install Dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/ajipal/ToneFit.git'
REPO_DIR = '/content/ToneFit'

if os.path.isdir(REPO_DIR):
    print('Repo already cloned. Fetching latest...')
    !cd {REPO_DIR} && git fetch origin && git checkout twelve-season && git pull origin twelve-season
else:
    !git clone -b twelve-season {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'✅ Working directory: {os.getcwd()}')
!ls

In [ ]:
!pip install -q timm>=0.9.0 scikit-image imagehash
print('✅ Dependencies installed')

---
## Step 3 — Link Dataset from Google Drive

> Update DRIVE_DATASET_PATH to where you uploaded RGB-M in Google Drive.

In [ ]:
import os

# ── UPDATE THIS PATH to where RGB-M is in your Google Drive ──
DRIVE_DATASET_PATH = '/content/drive/MyDrive/ToneFit/RGB-M'

# Create symlink so scripts can use RGB-M/ directly
LOCAL_LINK = '/content/ToneFit/RGB-M'
if not os.path.exists(LOCAL_LINK):
    os.symlink(DRIVE_DATASET_PATH, LOCAL_LINK)
    print(f'✅ Linked: {LOCAL_LINK} → {DRIVE_DATASET_PATH}')
else:
    print(f'✅ RGB-M already linked')

# Verify structure
print('\nDataset structure:')
for split in ['train', 'test']:
    split_path = os.path.join(LOCAL_LINK, split)
    if os.path.exists(split_path):
        seasons = [d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d))]
        total = 0
        print(f'  {split}/')
        for season in sorted(seasons):
            count = sum(len(files) for _, _, files in os.walk(os.path.join(split_path, season)))
            total += count
            print(f'    {season}/: {count} images')
        print(f'    TOTAL: {total} images')
    else:
        print(f'  ❌ {split}/ not found at {split_path}')

---
## Step 4 — Preprocessing (EDA Features Only)

> Extracts CIELab/HSV features from training images for EDA visualization.
> Does NOT re-split the dataset — RGB-M/train/ and RGB-M/test/ are used as-is.

In [ ]:
import sys
if 'preprocess' in sys.modules:
    del sys.modules['preprocess']

import preprocess
preprocess.run()
print('✅ Preprocessing complete')

In [ ]:
# Verify features.csv was created
import pandas as pd
df = pd.read_csv('features.csv')
print(f'features.csv: {len(df)} rows')
print('\nClass distribution:')
print(df['season'].value_counts())
print('\nSample:')
df.head()

---
## Step 5 — EDA

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

os.makedirs('results', exist_ok=True)
df = pd.read_csv('features.csv')

SEASONS = ['autumn', 'spring', 'summer', 'winter']
COLORS  = {'autumn': '#B7410E', 'spring': '#F4A261', 'summer': '#90B4CE', 'winter': '#4169E1'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('ToneFit ML — EDA Overview', fontsize=14, fontweight='bold')

# Class distribution
counts = df['season'].value_counts()[SEASONS]
axes[0].bar(counts.index, counts.values, color=[COLORS[s] for s in counts.index])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Images')
for i, (s, v) in enumerate(zip(counts.index, counts.values)):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# ITA Score by season
data = [df[df['season'] == s]['ITA'].values for s in SEASONS]
bp = axes[1].boxplot(data, labels=[s.capitalize() for s in SEASONS], patch_artist=True)
for patch, s in zip(bp['boxes'], SEASONS):
    patch.set_facecolor(COLORS[s])
    patch.set_alpha(0.7)
axes[1].set_title('ITA Score by Season\n(Higher = Lighter/Cooler)')
axes[1].set_ylabel('ITA Score')

# a* vs b* scatter (undertone)
for s in SEASONS:
    sub = df[df['season'] == s]
    axes[2].scatter(sub['a_mean'], sub['b_mean'], label=s.capitalize(),
                    color=COLORS[s], alpha=0.5, s=20)
axes[2].axhline(0, color='gray', ls='--', lw=0.8)
axes[2].axvline(0, color='gray', ls='--', lw=0.8)
axes[2].set_title('a* vs b* (Undertone)')
axes[2].set_xlabel('a* (warm/cool)')
axes[2].set_ylabel('b* (yellow/blue)')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('results/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA saved to results/eda_overview.png')

---
## Step 6 — Train FaRL-64 Baseline (Model A)

> Flat two-head classifier on frozen FaRL-64 features: jointly trains a 4-season head
> and a 12-subtype head on the same shared backbone representation.
>
> FaRL-64 pretrained weights must be in `models/farl_weights.pth`.
> Expected training time: ~20–30 minutes on T4 GPU.

In [ ]:
# Step 6.1 — Pull latest code, install CLIP, clear module cache
import sys, os

!cd /content/ToneFit && git pull
!pip install git+https://github.com/openai/CLIP.git -q

if 'train_farl' in sys.modules:
    del sys.modules['train_farl']

print('Setup complete')

In [ ]:
# Step 6.2 — Get FaRL pretrained weights (Drive first, then download)
import shutil, os
os.makedirs('models', exist_ok=True)

DRIVE_PATHS = [
    '/content/drive/MyDrive/ToneFit/farl_weights.pth',
    '/content/drive/MyDrive/ToneFit/models/farl_weights.pth',
]
LOCAL = 'models/farl_weights.pth'

if os.path.exists(LOCAL):
    print(f'FaRL weights already in models/ ({os.path.getsize(LOCAL)/1e6:.0f} MB)')
else:
    copied = False
    for src_path in DRIVE_PATHS:
        if os.path.exists(src_path):
            shutil.copy(src_path, LOCAL)
            print(f'Copied from Drive: {src_path}')
            copied = True
            break
    if not copied:
        print('Downloading FaRL weights (~652 MB)...')
        !wget -q --show-progress -O models/farl_weights.pth \
          "https://github.com/FacePerceiver/FaRL/releases/download/pretrained_weights/FaRL-Base-Patch16-LAIONFace20M-ep64.pth"
        drive_dst = DRIVE_PATHS[0]
        os.makedirs(os.path.dirname(drive_dst), exist_ok=True)
        shutil.copy(LOCAL, drive_dst)
        print(f'Saved to Drive: {drive_dst}')

print(f'Size: {os.path.getsize(LOCAL)/1e6:.0f} MB')

In [ ]:
# Step 6.3 — Train Model A (FaRL-64 Baseline: joint 4-season + 12-subtype)
import sys
if 'train_farl' in sys.modules:
    del sys.modules['train_farl']

import train_farl
train_farl.main()
print('Model A training complete')

In [ ]:
# Step 6.4 — Show Model A training curves
from IPython.display import Image as IPImage, display
import os

if os.path.exists('results/farl_training.png'):
    display(IPImage('results/farl_training.png'))
else:
    print('Training curves not found yet')

In [ ]:
# Step 6.5 — Save Model A results to Drive
import shutil, glob, os

DRIVE_MODELS  = '/content/drive/MyDrive/ToneFit/Evaluation/models'
DRIVE_RESULTS = '/content/drive/MyDrive/ToneFit/Evaluation/results/FARL'
os.makedirs(DRIVE_MODELS,  exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)

saved = []
if os.path.exists('models/farl_model.pth'):
    shutil.copy('models/farl_model.pth', DRIVE_MODELS + '/farl_model.pth')
    saved.append('models/farl_model.pth')

for f in glob.glob('results/farl*'):
    shutil.copy(f, DRIVE_RESULTS)
    saved.append(f)

for s in saved:
    print(f'Saved: {s}')
print(f'{len(saved)} file(s) saved to Drive')

---
## Step 7 — Train Hierarchical FaRL-64 (Novel Contribution)

> **Novel architecture:** frozen FaRL-64 backbone + two-stage hierarchical head.
> Stage 2 (Sub-Type) is conditioned on Stage 1 (Season) logits.
> Targets: Season acc > 0.554 (FaRL-64) AND Sub-Type acc > 0.318 (FaRL-16).
>
> Training saves a checkpoint every epoch.  If Colab crashes, re-run the
> "Check for resume checkpoint" cell and the training cell — it picks up
> exactly where it left off.
> Results are also synced to Drive every 5 epochs automatically.

In [ ]:
# Step 7.1 — Pull latest code and install pyyaml
!cd /content/ToneFit && git fetch origin && git checkout twelve-season && git pull origin twelve-season
!pip install -q pyyaml
import os
os.makedirs('results', exist_ok=True)
os.makedirs('models',  exist_ok=True)

files = ['train.py', 'hierarchical_head.py', 'configs/hierarchical.yaml']
for f in files:
    exists = os.path.exists(f)
    print(f"{'OK' if exists else 'MISSING'} {f}")
print('Setup complete')

In [ ]:
import os, shutil

DRIVE_FARL  = '/content/drive/MyDrive/ToneFit/Evaluation/models/farl_model.pth'
LOCAL_FARL  = 'models/farl_model.pth'

if not os.path.exists(LOCAL_FARL):
    if os.path.exists(DRIVE_FARL):
        shutil.copy(DRIVE_FARL, LOCAL_FARL)
        size = os.path.getsize(LOCAL_FARL) / 1e6
        print(f'Copied farl_model.pth from Drive ({size:.0f} MB)')
    else:
        print('WARNING: farl_model.pth not found in Drive or local.')
        print('Training will start from random ViT-B/16 weights (still valid).')
else:
    size = os.path.getsize(LOCAL_FARL) / 1e6
    print(f'farl_model.pth already present ({size:.0f} MB)')

In [ ]:
import glob, os, re

RESULTS_DIR = 'results'
DRIVE_CKPT_DIR = '/content/drive/MyDrive/ToneFit/results'

# If results/ is empty, try to restore checkpoints from Drive first
local_ckpts = glob.glob(f'{RESULTS_DIR}/checkpoint_epoch_*.pth')
if not local_ckpts and os.path.isdir(DRIVE_CKPT_DIR):
    drive_ckpts = glob.glob(f'{DRIVE_CKPT_DIR}/checkpoint_epoch_*.pth')
    if drive_ckpts:
        print(f'Restoring {len(drive_ckpts)} checkpoint(s) from Drive...')
        for f in drive_ckpts:
            shutil.copy(f, RESULTS_DIR)
        # Also restore best checkpoint and history if present
        for fname in ('best_hierarchical.pth', 'hierarchical_history.json'):
            src = os.path.join(DRIVE_CKPT_DIR, fname)
            if os.path.exists(src):
                shutil.copy(src, RESULTS_DIR)
        local_ckpts = glob.glob(f'{RESULTS_DIR}/checkpoint_epoch_*.pth')

# Find the highest epoch checkpoint
def epoch_num(path):
    m = re.search(r'checkpoint_epoch_(\d+)\.pth', path)
    return int(m.group(1)) if m else -1

resume_arg = ''
if local_ckpts:
    latest = max(local_ckpts, key=epoch_num)
    ep = epoch_num(latest)
    resume_arg = f'--resume {latest}'
    print(f'Found checkpoint at epoch {ep}. Will resume from: {latest}')
else:
    print('No checkpoint found. Starting fresh training.')

print(f'\nresume_arg = {resume_arg!r}')

In [ ]:
import subprocess, sys

DRIVE_RESULTS = '/content/drive/MyDrive/ToneFit/results'
CONFIG        = 'configs/hierarchical.yaml'

cmd = (
    f'{sys.executable} train.py '
    f'--config {CONFIG} '
    f'--drive_dir {DRIVE_RESULTS} '
    f'{resume_arg}'
).strip()

print('Running:', cmd)
print('=' * 65)

# Run with live output
proc = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode == 0:
    print('\nTraining complete.')
else:
    print(f'\nProcess exited with code {proc.returncode}')

In [ ]:
import json, os
from IPython.display import Image as IPImage, display

# Print best result from history
hist_path = 'results/hierarchical_history.json'
if os.path.exists(hist_path):
    history = json.load(open(hist_path))
    best = max(history, key=lambda r: r.get('val_season_acc', 0))
    print(f"Best epoch : {best['epoch']}")
    print(f"Season acc : {best['val_season_acc']:.4f}  (target > 0.554)")
    print(f"Subtype acc: {best['val_subtype_acc']:.4f}  (target > 0.318)")
    beat_s   = best['val_season_acc']  > 0.554
    beat_sub = best['val_subtype_acc'] > 0.318
    print(f"Beat FaRL-64 season?    {'YES' if beat_s   else 'not yet'}")
    print(f"Beat FaRL-16 sub-type?  {'YES' if beat_sub else 'not yet'}")
else:
    print('No history file yet.')

---
## Step 8 — Compare All Models

> Runs `evaluate.py` to produce a side-by-side comparison table.
> Run AFTER Step 6 and Step 7 are complete.
> Outputs: `results/evaluation_summary.csv` and `results/evaluation_summary.txt`

In [ ]:
# Step 8 — Run evaluate.py to compare all trained models
import sys
if 'evaluate' in sys.modules:
    del sys.modules['evaluate']

import evaluate
evaluate.main()

---
## Step 9 — Save Results to Drive

In [ ]:
import shutil, glob, os

DRIVE_RESULTS = '/content/drive/MyDrive/ToneFit/results'
DRIVE_MODELS  = '/content/drive/MyDrive/ToneFit/models'
os.makedirs(DRIVE_RESULTS, exist_ok=True)
os.makedirs(DRIVE_MODELS,  exist_ok=True)

for f in glob.glob('results/*'):
    shutil.copy(f, DRIVE_RESULTS)
print(f'Results saved to Drive: {DRIVE_RESULTS}')

for f in glob.glob('models/*.pth'):
    shutil.copy(f, DRIVE_MODELS)
if os.path.exists('farl_model.pth'):
    shutil.copy('farl_model.pth', DRIVE_MODELS)
print(f'Models saved to Drive: {DRIVE_MODELS}')

if os.path.exists('features.csv'):
    shutil.copy('features.csv', '/content/drive/MyDrive/ToneFit/features.csv')
    print('features.csv saved to Drive')

In [ ]:
import zipfile, glob, os
from google.colab import files

zip_path = 'ToneFit_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob('results/*'):
        z.write(f)
    for f in glob.glob('models/*.pth'):
        z.write(f)
    if os.path.exists('farl_model.pth'):
        z.write('farl_model.pth')
    if os.path.exists('features.csv'):
        z.write('features.csv')

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f'Created: {zip_path} ({size_mb:.1f} MB)')
files.download(zip_path)